# Various Plots to analyse influence of rain

In [ ]:
from tuecycle.utils.transforms import (
    add_time_features,      # Adds hour, dayofweek, month, year_month, is_weekend
    filter_daytime,         # Filters to hours 6-22
    compute_deviations,     # Adds temp_deviation and bike_deviation columns
    classify_time_category, # Adds time_category (rush hour classification)
    add_season,             # Adds season column (Winter/Transition/Summer)
    prepare_fft_data,       # Prepares data for FFT analysis
)
from tuecycle import DataManager
import pandas as pd
from tuecycle.config.stations import get_stations_by_city
from tuecycle.plots import get_plot
from typing import Union, Sequence


1. Set the time window and cities

In [ ]:
START_DATE = pd.Timestamp(2020, 1, 1)
END_DATE   = pd.Timestamp(2025, 11, 30)

dm = DataManager(
    base_path="..",
    start_date=(START_DATE.year, START_DATE.month, START_DATE.day),
    end_date=(END_DATE.year, END_DATE.month, END_DATE.day),
)

cities = ["mannheim", "tuebingen", "heidelberg"]



### 1. Preprocessing

Filter out stations that did not exist at the time of the start date. Preprocess and add necessary features

In [ ]:
data_dict = {}
city_data_dict = {}

for city in cities:
    stations = get_stations_by_city(city)
    dfs_city = []

    for station in stations:
        # Does the data exist at all or is it faulty?
        try:
            df = dm.get(station.alias)
        except FileNotFoundError:
            #print(f"Skipping {station.alias}, bike or weather data missing.")
            continue
        except Exception as e:
            #print(f"Skipping {station.alias}, error: {e}")
            continue

        # Does the data exist for the whole time window?
        if df['datetime'].max() < START_DATE:
            #print(f"Skipping {station.alias}, no data since {START_DATE.date()}")
            continue

        # Add time features
        df = add_time_features(df)
        df = classify_time_category(df)
        
        data_dict[station.alias] = df
        dfs_city.append(df)

    if dfs_city:
        city_data_dict[city] = pd.concat(dfs_city, ignore_index=True)

### 2. Define several functions for plotting

In [ ]:
from typing import Union, Sequence
import pandas as pd

def city_rain_shares(city_data_dict, temp_max, temp_min, hours: Union[str, Sequence[str], None] = None):
    """
    Calculates the percentage of bicycle journeys in rainy weather per city,
    optionally split by time categories.

    Args:
        city_data_dict: dict {city_name: DataFrame of all stations in given city}
        temp_max: max temperature [°C]
        temp_min: min temperature [°C]
        hours:  - 'Morning Rush (7-9)': Weekday hours 7-8
                - 'Evening Rush (17-19)': Weekday hours 17-18
                - 'Weekday Non-Rush': Other weekday hours
                - 'Weekend': Saturday and Sunday

    Returns:
        pd.DataFrame with ["city", "time_category", "rain_share"]
    """
    records = []

    for city, df in city_data_dict.items():
        df_analysis = df[
            (df['temp'] <= temp_max) &
            (df['temp'] >= temp_min)
        ]

        if df_analysis.empty or df_analysis['bike'].sum() == 0:
            continue

        # Only keep requested hours if specified
        if hours is not None:
            if isinstance(hours, str):
                hours = [hours]
            df_analysis = df_analysis[df_analysis['time_category'].isin(hours)]

        if df_analysis.empty:
            continue

        # Compute rain share per time_category
        for cat, group in df_analysis.groupby('time_category'):
            total_bikes = group['bike'].sum()
            if total_bikes == 0:
                continue
            rain_bikes = group[group['rain'] > 0]['bike'].sum()
            rain_share = rain_bikes / total_bikes

            records.append({
                "city": city.capitalize(),
                "time_category": cat,
                "rain_share": rain_share
            })

    return pd.DataFrame(records)


In [ ]:
def filter_rain(base_data: dict, temp_max: float = 50, temp_min: float = -30,
                     hours: Union[str, Sequence[str], None] = None
) -> dict:
    """
    Filters bike traffic data for rainy conditions during rush hours and normalizes bike counts.

    Args:
        base_data (dict)
        temp_max (float, optional): max. temperature [°C]
        temp_min (float, optional): min. temperature [°C]
        hours:  - 'Morning Rush (7-9)': Weekday hours 7-8
                - 'Evening Rush (17-19)': Weekday hours 17-18
                - 'Weekday Non-Rush': Other weekday hours
                - 'Weekend': Saturday and Sunday

    Returns:
        dict: Filtered dictionary with bike counts normalized.
    """
    filtered = {}

    for alias, df in base_data.items():
        df = df.copy()

        # Min-Max Normalization of bike counts per station

        bike_min = df['bike'].min()
        bike_max = df['bike'].max()
        if bike_max > bike_min:
            df['bike'] = (df['bike'] - bike_min) / (bike_max - bike_min)
        else:
            df['bike'] = 0.0  # If all counts are the same, set normalized value to 0

        # Filter data for analysis
        df_analysis = df[
            (df['temp'] <= temp_max) &
            (df['temp'] >= temp_min) &
            (df['rain'] > 0) 
        ]

        if hours is not None:
            if isinstance(hours, str):
                hours = [hours]  # convert single string to list
            df_analysis = df_analysis[df_analysis['time_category'].isin(hours)]



        if not df_analysis.empty:
            filtered[alias] = df_analysis

    return filtered


### 6. Plot using the filter functions

- Figure 1 - 2 : Number of cyclists in the rain at different temperature (2019-2025) during rush hour and weekend 
  Default is no hour filter.

- Figure 3: Share of cyclists in the rain for each city at ≤5°C (2019-2025) during different hours

In [ ]:
fig1 = get_plot("bike_vs_rain_rush_hour_city")(
    filter_rain(data_dict, temp_max =5, hours=['Morning Rush (7-9)', 'Evening Rush (17-19)']),
    cities=cities,
    title="Number of cyclists in the rain at ≤5°C (2019-2025) - Rush Hour"
)
fig1.show()

fig2 = get_plot("bike_vs_rain_rush_hour_city")(
    filter_rain(data_dict, temp_max = 5, hours=['Weekend']),
    cities=cities,
    title="Number of cyclists in the rain at ≤5°C (2019-2025) - Weekend"
)
fig2.show()

fig3 = get_plot("city_rain_share_boxplot")(
    city_rain_shares(city_data_dict, temp_max=5, temp_min=-30),
    title="Rain share ≤5°C"
)
fig3.show()

